In [1]:
import os
from utils.gmat import gmat, gmat_path

In [2]:
kwargs = {}
coe = [9750.5, 0.7, 63.5, 90.0, 0.0, 30.0]
earth_potential_file_path = os.path.join(
    gmat_path, "data", "gravity", "earth", "JGM3.cof"
)
luna_potential_file_path = os.path.join(
    gmat_path, "data", "gravity", "luna", "grgm900c.cof"
)

EarthMJ2000Eq = gmat.Construct(
    "CoordinateSystem", "EarthMJ2000Eq", "Earth", "MJ2000Eq")
LunaMJ2000Eq = gmat.Construct(
    "CoordinateSystem", "LunaMJ2000Eq", "Luna", "MJ2000Eq")

# Spacecraft configuration preliminaries
sc = gmat.Construct("Spacecraft", "LunaOrbiter")
sc.SetField("DateFormat", "UTCGregorian")
sc.SetField("Epoch", "20 Jul 2020 12:00:00.000")
sc.SetField("CoordinateSystem", "LunaMJ2000Eq")
# sc.SetField("DisplayStateType", "Keplerian")

# Orbital state
sc.SetField("SMA", coe[0])
sc.SetField("ECC", coe[1])
sc.SetField("INC", coe[2])
sc.SetField("RAAN", coe[3])
sc.SetField("AOP", coe[4])
sc.SetField("TA", coe[5])

# Spacecraft ballistic properties for the SRP and Drag models
if "SRPArea" in kwargs:
    sc.SetField("SRPArea", 2.5)
if "Cr" in kwargs:
    sc.SetField("Cr", 1.75)
if "DragArea" in kwargs:
    sc.SetField("DragArea", 1.8)
if "Cd" in kwargs:
    sc.SetField("Cd", 2.1)

sc.SetField("DryMass", 80)

# Force model settings
fm = gmat.Construct("ForceModel", "FM")
fm.SetField("CentralBody", "Luna")

# An 8x8 JGM-3 Gravity Model
grav = gmat.Construct("GravityField")
grav.SetField("BodyName", "Luna")
grav.SetField("PotentialFile", luna_potential_file_path)
grav.SetField("Degree", 5)
grav.SetField("Order", 5)

# Add forces into the ODEModel container
fm.AddForce(grav)

gmat.Initialize()

# Build the propagation container class
pdprop = gmat.Construct("Propagator", "PDProp")

# Create and assign a numerical integrator for use in the propagation
gator = gmat.Construct("PrinceDormand78", "Gator")
pdprop.SetReference(gator)

# Set some of the fields for the integration
pdprop.SetField("InitialStepSize", 60.0)
pdprop.SetField("Accuracy", 1.0e-12)
pdprop.SetField("MinStep", 0.0)

# Perform top level initialization
gmat.Initialize()

# Setup the spacecraft that is propagated
pdprop.AddPropObject(sc)
pdprop.PrepareInternals()

# Refresh the 'gator reference
fm = pdprop.GetODEModel()
gator = pdprop.GetPropagator()

# Take a 60 second step, showing the state before and after propagation
print(sc.GetEpoch())
print(gator.GetState())
print(sc.GetState().GetState())
print(sc.GetCartesianState())

29051.000428638676
[-160383.0872801545, 310750.9441030899, 151582.86791315538, -1.6148863649767295, -0.9745829992757649, 1.2747101644119958]
[-160383.0872801545, 310750.9441030899, 151582.86791315538, -1.6148863649767295, -0.9745829992757649, 1.2747101644119958]
-690.7009840444371 2681.16329137719 1385.331855182332 -0.6938248067068136 -0.4964706593589572 1.39159727414666


In [6]:
grav.GetDerivatives(sc.GetState().GetState())

APIException: ODEModel Exception Thrown: GravityField requires at least one spacecraft.

In [ ]:
print(grav.GetDerivativesForSpacecraft(sc))

APIException: ODEModel Exception Thrown: GravityField requires at least one spacecraft.